# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [14]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [1]:
import pandas as pd
from datasets import Dataset, load_dataset

print("Pandas and Dataset utilities imported successfully.")

Pandas and Dataset utilities imported successfully.


**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [2]:
import pandas as pd
from datasets import Dataset

# Load the UCI SMS Spam dataset from the provided parquet link
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)

# Convert the pandas DataFrame into a Hugging Face Dataset object
hf_dataset = Dataset.from_pandas(df)

# Define splits: 4,000 for training and 1,000 for validation (total 5,000)
TRAIN_START = 0
TRAIN_END = 4000
VAL_START = 4000
VAL_END = 5000

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))

print(f"Training samples: {len(train_ds)}, Validation samples: {len(val_ds)}")
display(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Training samples: 4000, Validation samples: 1000


,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [3]:
from transformers import GPT2Tokenizer

# Use the base GPT-2 model
MODEL_NAME = 'gpt2'

# Initialize the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# GPT-2 lacks a dedicated pad token; we use the End-Of-Sequence token instead
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer initialized with pad_token: {tokenizer.pad_token}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer initialized with pad_token: <|endoftext|>


In [4]:
TEXT_COLUMN = 'sms'
PADDING_STRATEGY = 'max_length'
TRUNCATION_FLAG = True
MAX_SEQ_LEN = 64

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )

# Map the tokenization function across our datasets
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

print("Tokenization complete for both splits.")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenization complete for both splits.


# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [5]:
import torch
from transformers import GPT2ForSequenceClassification

# Binary classification: Spam (1) vs Ham (0)
NUM_LABELS = 2

# Load pre-trained GPT-2 with a classification head
model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,
)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Model loaded and moved to {device}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to cuda


# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [6]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Scaling factor 1/sqrt(d_k) to prevent vanishing/exploding gradients in softmax
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        # Compute dot product between Q and K^T, then scale
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Softmax over the last dimension to get weights
        attn = F.softmax(scores, dim=-1)

        # Weighted sum of values
        return torch.matmul(attn, value), attn

class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x shape: [batch, seq_len]
        embed = self.embedding(x) # [batch, seq_len, embed_dim]

        # Self-attention: Q, K, and V all come from the same embedding
        attn_output, _ = self.attn(embed, embed, embed)

        # Global Average Pooling over the sequence (dimension 1)
        pooled = attn_output.mean(dim=1)

        # Classification head
        return self.fc(pooled)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [7]:
ATTN_TEXT_COLUMN = 'sms'
ATTN_MAX_LEN = 64

def preprocess_for_attention(example):
    # Manual encoding to ensure labels are kept in the returned dictionary
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

print("Custom model preprocessing complete.")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Custom model preprocessing complete.


In [8]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }

# Create DataLoaders for PyTorch training
train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

print("DataLoaders ready.")

DataLoaders ready.


In [9]:
# derive vocab size from tokenizer
vocab_size = tokenizer.vocab_size
embed_dim = 64
num_classes = 2
learning_rate = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Simple training loop
attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print(f'Custom Attention model trained. Final batch loss: {loss.item():.4f}')

Custom Attention model trained. Final batch loss: 0.2571


# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [13]:
try:
    import evaluate
except ImportError:
    !pip install evaluate
    import evaluate

import numpy as np

# Load standard classification metrics
accuracy = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall = evaluate.load('recall')
f1 = evaluate.load('f1')

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall': recall.compute(predictions=preds, references=labels)['recall'],
        'f1': f1.compute(predictions=preds, references=labels)['f1'],
    }

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [15]:
print("\n📊 Evaluating GPT-2 Model...")
gpt2_preds = []
gpt2_labels = []
model.eval()

for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)


📊 Evaluating GPT-2 Model...
GPT-2 Metrics: {'accuracy': 0.803, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


In [16]:
print("\n📊 Evaluating Custom Attention Model...")
attn_preds = []
attn_labels = []
attn_model.eval()

for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1'],
}
print('Attention Model Metrics:', attn_metrics)


📊 Evaluating Custom Attention Model...
Attention Model Metrics: {'accuracy': 0.859, 'precision': 0.42857142857142855, 'recall': 0.04316546762589928, 'f1': 0.0784313725490196}


# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?
- **Query (Q):** Represents the current item looking for information. It asks "What am I looking for?"
- **Key (K):** Represents the information index or labels for all items. It helps determine the relevance of each item to the query.
- **Value (V):** Contains the actual content or information. Once the relevance (attention weights) is calculated using Q and K, we multiply those weights by V to get the final context-aware representation.

### 2. Why do we use a scaling factor in the dot-product attention?
We use a scaling factor (typically $1/\sqrt{d_k}$) because as the dimensionality of the embeddings ($d_k$) increases, the dot products tend to grow very large in magnitude. Large values can push the softmax function into regions where gradients are extremely small (vanishing gradients), making training difficult. Scaling keeps the dot products in a range that maintains stable gradients.

### 3. How does self-attention differ from traditional sequence models like RNNs?
- **Parallel vs. Sequential:** RNNs process tokens one by one (linearly), making them slow for long sequences. Self-attention processes all tokens simultaneously, allowing for massive parallelization.
- **Dependency Range:** RNNs often struggle with long-range dependencies due to the bottleneck of the hidden state. Self-attention has a "receptive field" that covers the entire sequence instantly, meaning the distance between tokens doesn't hinder communication.
- **Complexity:** Self-attention has $O(n^2 \cdot d)$ complexity relative to sequence length $n$, while RNNs are $O(n \cdot d^2)$.

### 4. Performance analysis
- **Comparison:** In this experiment, the **Custom Attention Model** actually performed better in terms of Accuracy (~85.9%) compared to the base **GPT-2** (~80.3%) which was not fully fine-tuned here. However, both models showed very low Recall, meaning they struggle to correctly identify the 'Spam' class because the dataset is likely imbalanced (mostly 'Ham').
- **Trade-offs:** GPT-2 is massive and requires significant memory but has a deep understanding of language. The custom model is extremely lightweight and fast to train but lacks the nuance of pre-trained knowledge.
- **Improvement:** To improve the custom model, we could add **Multi-Head Attention** (to capture different types of relationships) or implement a **Class Weighting** strategy in the Loss function to help the model better recognize the minority Spam class.